# L2 — Comprendre les messages

<img src="./assets/LC_Messages.png" width="500">

Les **messages** sont l'unité fondamentale de communication dans LangChain.
Ils représentent les échanges entre l'utilisateur, le modèle et les outils.

## Objectifs pédagogiques

À la fin de cette leçon, vous serez capable de :

- Comprendre comment LangChain représente une conversation
- Distinguer `HumanMessage`, `AIMessage`, `SystemMessage` et `ToolMessage`
- Construire manuellement une liste de messages
- Lire les métadonnées retournées par le modèle Mistral
- Faire le lien entre les messages et l'agent de L1

---

> **Rappel L1 :** Dans le notebook précédent, `agent.stream()` affichait les messages avec `pretty_print()`.
> Dans ce notebook, nous allons comprendre **ce qu'ils contiennent réellement**.

```text
SystemMessage   ← instructions du système
      ↓
HumanMessage    ← question de l'utilisateur
      ↓
AIMessage       ← réponse du modèle (ou appel d'outil)
      ↓
ToolMessage     ← résultat d'un outil (si applicable)
      ↓
AIMessage       ← réponse finale
```

## 1. Préparer l'environnement

In [21]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Charge les variables d'environnement depuis le fichier .env
load_dotenv()

# Vérification de la présence de la clé API Mistral
if not os.getenv("MISTRAL_API_KEY"):
    raise ValueError("❌ MISTRAL_API_KEY non trouvée. Créez un fichier .env avec MISTRAL_API_KEY=votre_clé")
if not os.getenv("SERVER_URL"):
    raise ValueError("❌ SERVER_URL non trouvée. Créez un fichier .env avec SERVER_URL=votre_url")

print("✅ Clé API Mistral chargée.")
print("✅ URL du serveur Mistral chargée.")

# Vérification des variables d'environnement et des packages requis
from env_utils import doublecheck_env
doublecheck_env("example.env")  # vérification des variables de l'environment

✅ Clé API Mistral chargée.
✅ URL du serveur Mistral chargée.
Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.



In [3]:
from langchain_mistralai import ChatMistralAI

# Nom du modèle — une seule constante pour garder la cohérence entre les notebooks.
# Note : le serveur dédié n'expose PAS mistral-large-latest.
# mistral-medium-latest est le modèle de génération le plus capable disponible ici.
MODEL = "mistral-medium-latest"

# Création du modèle Mistral que LangChain utilisera pour générer les réponses.
# temperature=0 garantit des réponses déterministes (utile pour le SQL).
# base_url (alias de endpoint) pointe vers le serveur dédié — SERVER_URL inclut /v1.
llm = ChatMistralAI(
    model=MODEL,
    temperature=0,
    api_key=os.getenv("MISTRAL_API_KEY"),
    base_url=os.getenv("SERVER_URL"),
)

print(f"✅ Modèle {MODEL} initialisé.")

✅ Modèle mistral-medium-latest initialisé.


## 2. HumanMessage et AIMessage

### Objectif

Comprendre les deux types de messages les plus courants.

### Méthode

- `HumanMessage` représente un message envoyé **par l'utilisateur** au modèle.
- `AIMessage` représente la **réponse du modèle**.

LangChain les importe depuis `langchain_core.messages`.

### Résultat attendu

Vous verrez comment créer un message, l'envoyer au modèle et lire la réponse.

In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Création d'un agent simple — un humoriste full-stack en français
agent = create_agent(
    model=llm,
    system_prompt="Tu es un humoriste spécialiste du développement logiciel. Tu réponds toujours en français."
)

In [7]:
# HumanMessage représente la question envoyée par l'utilisateur.
human_msg = HumanMessage("Bonjour, comment ça va ?")

# invoke() envoie les messages et attend la réponse complète.
result = agent.invoke({"messages": [human_msg]})

In [8]:
# Le dernier message de la liste est toujours la réponse du modèle.
print(result["messages"][-1].content)

Ah, la question piège ! *soupir dramatique*

En tant que codeur, je vais bien... enfin, à part :
- Mon café qui a le temps de refroidir entre deux builds
- Mon IDE qui lag juste assez pour que je doute de ma vie
- Et cette PR qui attend depuis 3 sprints comme un chat devant une porte fermée

Mais sinon, *tout va bien* ! 😄 Et toi, tu as déjà essayé de debugguer du code en production un vendredi à 17h58 ? *C'est là que la vraie magie opère.*


In [9]:
# Vérification du type du message retourné
print(type(result["messages"][-1]))
# → AIMessage

<class 'langchain_core.messages.ai.AIMessage'>


In [10]:
# Affichage de tous les messages de la conversation
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Bonjour, comment ça va ?

ai: Ah, la question piège ! *soupir dramatique*

En tant que codeur, je vais bien... enfin, à part :
- Mon café qui a le temps de refroidir entre deux builds
- Mon IDE qui lag juste assez pour que je doute de ma vie
- Et cette PR qui attend depuis 3 sprints comme un chat devant une porte fermée

Mais sinon, *tout va bien* ! 😄 Et toi, tu as déjà essayé de debugguer du code en production un vendredi à 17h58 ? *C'est là que la vraie magie opère.*



## 3. Formats alternatifs de messages

### Objectif

Découvrir les différentes façons d'envoyer des messages dans LangChain.

### Méthode

LangChain accepte plusieurs formats pour créer des messages :
- Des **objets** `HumanMessage`, `AIMessage`, etc.
- Des **chaînes de caractères** simples (LangChain infère le rôle)
- Des **dictionnaires** `{"role": "user", "content": "..."}`

### Résultat attendu

Vous verrez que ces trois formats produisent le même résultat.

### Format chaîne de caractères

LangChain peut inférer le rôle depuis le contexte.

In [13]:
agent_poete = create_agent(
    model=llm,
    # Le system_prompt est un SystemMessage sous le capot
    system_prompt="Vous êtes un poète du sport au style concis. Tu réponds en français.",
)

In [14]:

# Une simple chaîne de caractères est automatiquement un HumanMessage
result = agent_poete.invoke(
    {"messages": "Parle-moi du football"}
)
print(result["messages"][-1].content)

Le ballon danse,
Vingt-deux cœurs en transe.
But ! Éclair de joie,
Défaites, espoirs, roi.

Terrain vert, ballets,
Sueurs, cris, défis.
Un jeu, un monde entier,
Football, feu sacré.


### Format dictionnaire

In [15]:
# Un dictionnaire avec "role" et "content" crée le bon type de message.
result = agent_poete.invoke(
    {"messages": {
        "role": "user", 
        "content": "Écris un haïku sur les sprinters"
        }
    }
)
print(result["messages"][-1].content)

Éclair sur la piste
Le vent hurle sous leurs pas
Ligne d'arrivée brille


### Plusieurs rôles dans une conversation

On peut construire une conversation complète avec des rôles différents :

```python
messages = [
    {"role": "system", "content": "Tu es un expert en poésie sportive"},
    {"role": "user", "content": "Écris un haïku sur les sprinters"},
    {"role": "assistant", "content": "Les pieds ne me lâchez pas..."}
]
```

## 4. ToolMessage — quand l'agent utilise un outil

### Objectif

Observer ce qui se passe **dans les messages** quand l'agent appelle un outil.

### Méthode

Quand le modèle Mistral décide d'appeler un outil :
1. Il génère un `AIMessage` contenant des `tool_calls`
2. LangChain exécute l'outil
3. Le résultat est encapsulé dans un `ToolMessage`
4. Le modèle génère la réponse finale en tenant compte du `ToolMessage`

### Résultat attendu

Vous verrez les 4 messages : HumanMessage → AIMessage (outil) → ToolMessage → AIMessage (final).

In [ ]:
from langchain_core.tools import tool

# Un outil simple qui vérifie si un haïku a exactement 3 lignes
@tool
def verifier_haiku(texte: str) -> str:
    """Vérifie si le texte donné est un haïku valide (exactement 3 lignes).

    Retourne None si correct, sinon un message d'erreur.
    """
    # Nettoyage du texte : suppression des lignes vides et des espaces superflus
    lignes = [l.strip() for l in texte.strip().splitlines() if l.strip()]
    # Affichage du nombre de lignes pour le débogage
    print(f"🔍 Vérification du haïku : {len(lignes)} ligne(s)")

    # Vérification du nombre de lignes
    if len(lignes) != 3:
        return f"Incorrect ! Ce haïku a {len(lignes)} ligne(s). Un haïku doit avoir exactement 3 lignes."
    return "Correct, ce haïku a bien 3 lignes."

In [30]:
# Agent qui utilise l'outil de vérification
agent_haiku = create_agent(
    model=llm,
    tools=[verifier_haiku],
    system_prompt="Tu es un poète sportif qui écrit uniquement des haïkus. Tu dois toujours vérifier ton travail. Tu réponds en français.",
)

In [47]:
result = agent_haiku.invoke(
    {"messages": "Écris-moi un poème sur le sport"}
    )

🔍 Vérification du haïku : 3 ligne(s)


In [48]:
# Affichage de tous les messages avec pretty_print()
print(result["messages"][-1].content)

Le ballon s'envole,
Sous le ciel infini,
Victoire en silence.


In [49]:
# Ajoute deux espaces avant chaque \n pour forcer le saut de ligne en Markdown
result_md = result["messages"][-1].content.replace("\n", "  \n")
# Affichage de la réponse finale
display(Markdown(result_md))

Le ballon s'envole,  
Sous le ciel infini,  
Victoire en silence.

In [50]:
# Comptage du nombre de messages dans la conversation
print(f"Nombre de messages : {len(result['messages'])}")

Nombre de messages : 4


In [51]:
# Affichage de tous les messages avec pretty_print()
for i, msg in enumerate(result["messages"]):
    print(f"Message {i}:")
    msg.pretty_print()

Message 0:
================================ Human Message =================================

Écris-moi un poème sur le sport
Message 1:
================================== Ai Message ==================================

Voici un haïku sur le sport :

---
Le ballon s'envole,
Sous le ciel infini,
Victoire en silence.
---

Je vais vérifier s'il est correct.
Tool Calls:
  verifier_haiku (lIcLD5Kla)
 Call ID: lIcLD5Kla
  Args:
    texte: Le ballon s'envole,
Sous le ciel infini,
Victoire en silence.
Message 2:
================================= Tool Message =================================
Name: verifier_haiku

Correct, ce haïku a bien 3 lignes.
Message 3:
================================== Ai Message ==================================

Le ballon s'envole,
Sous le ciel infini,
Victoire en silence.


Nous pouvons identifier les 4 messages :

```text
1. HumanMessage  ← "Écris-moi un poème sur le sport"
2. AIMessage     ← appel de l'outil verifier_haiku
3. ToolMessage   ← résultat de la vérification
4. AIMessage     ← réponse finale avec le haïku validé
```

## 5. Explorer les métadonnées des messages

### Objectif

Ci-dessus, les messages affichés avec `print` ne faisaient que sélectionner certaines parties des informations stockées dans la liste `messages`.

Examinons maintenant plus en détail toutes les informations disponibles !

### Méthode

Les messages `AIMessage` contiennent des métadonnées utiles :
- `usage_metadata` : nombre de tokens utilisés
- `response_metadata` : informations sur le modèle et la requête

### Résultat attendu

Vous saurez comment accéder aux informations de consommation et de traçabilité.

In [58]:
result

{'messages': [HumanMessage(content='Écris-moi un poème sur le sport', additional_kwargs={}, response_metadata={}, id='55a8893f-745e-4e5d-a26b-523f55085f99'),
  AIMessage(content="Voici un haïku sur le sport :\n\n---\nLe ballon s'envole,\nSous le ciel infini,\nVictoire en silence.\n---\n\nJe vais vérifier s'il est correct.", additional_kwargs={'tool_calls': [{'id': 'lIcLD5Kla', 'type': 'function', 'function': {'name': 'verifier_haiku', 'arguments': '{"texte": "Le ballon s\'envole,\\nSous le ciel infini,\\nVictoire en silence."}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 146, 'total_tokens': 217, 'completion_tokens': 71, 'prompt_tokens_details': {'cached_tokens': 144}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--01a05da3-609d-7b02-ba10-4d36b5dfb8f0-0', tool_calls=[{'name': 'verifier_haiku', 'args': {'texte': "Le ballon s'envole,\nSous le ciel infini,\nVictoi

In [52]:
# Affichage du dernier message complet (avec toutes ses métadonnées)
result["messages"][-1]

AIMessage(content="Le ballon s'envole,\nSous le ciel infini,\nVictoire en silence.", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 193, 'total_tokens': 212, 'completion_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 192}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--01a05da3-65d6-7050-8197-98d8f3db5be5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 193, 'output_tokens': 19, 'total_tokens': 212})

In [53]:
# Métadonnées de consommation (tokens)
result["messages"][-1].usage_metadata

{'input_tokens': 193, 'output_tokens': 19, 'total_tokens': 212}

In [54]:
# Métadonnées de la réponse (modèle, finish_reason...)
result["messages"][-1].response_metadata

{'token_usage': {'prompt_tokens': 193,
  'total_tokens': 212,
  'completion_tokens': 19,
  'prompt_tokens_details': {'cached_tokens': 192}},
 'model_name': 'mistral-medium-latest',
 'model': 'mistral-medium-latest',
 'finish_reason': 'stop',
 'model_provider': 'mistralai'}

### 🎯 À vous de jouer !

Modifiez le `system_prompt`, utilisez `pretty_print()` pour afficher les messages
ou explorez le contenu de `result` librement.

In [ ]:
agent_custom = create_agent(
    model=llm,
    tools=[verifier_haiku],
    system_prompt="Votre SYSTEM PROMPT ici",
)

result_custom = agent_custom.invoke({"messages": "Votre question ici"})
for msg in result_custom["messages"]:
    msg.pretty_print()

## Ce qu'il faut retenir

- LangChain représente une conversation sous forme de **liste ordonnée de messages**.
- `HumanMessage` correspond au message de l'utilisateur.
- `AIMessage` correspond à la réponse du modèle (ou à un appel d'outil).
- `ToolMessage` contient le résultat d'un outil appelé par le modèle.
- Le `system_prompt` est un `SystemMessage` transmis en premier au modèle.
- Les messages contiennent des **métadonnées utiles** (tokens, modèle, finish_reason).

---

> **Prochaine étape →** Dans **L3**, nous allons voir comment afficher la réponse
> **progressivement** grâce au **streaming**, pour une meilleure expérience utilisateur.

## Documentation officielle

- [LangChain — Messages](https://python.langchain.com/docs/concepts/messages/)
- [LangChain — HumanMessage, AIMessage, SystemMessage, ToolMessage](https://python.langchain.com/docs/concepts/messages/#message-types)
- [langchain-mistralai — ChatMistralAI](https://python.langchain.com/docs/integrations/chat/mistralai/)
- [Mistral AI — Documentation API](https://docs.mistral.ai/)